# Intro to PyTorch

PyTorch is a framework for deep learning methods in Python. It is used to create, train, and evaluate neural networks.



In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform

# Device selection: CUDA → MPS (Apple Silicon) → CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device.type}\n")

if device.type == "cuda":
    num_gpus = torch.cuda.device_count()
    print(f"Found {num_gpus} CUDA device(s):")
    for idx in range(num_gpus):
        print(f"  [{idx}] {torch.cuda.get_device_name(idx)}")
elif device.type == "mps":
    print("Apple Metal (MPS) device available")
else:
    print("Running on CPU")

print("\nPython:", platform.python_version(), "| PyTorch:", torch.__version__)

# Tensors

Tensors are the fundamental data type in PyTorch and are very similar to NumPy arrays. They can handle the same operations as arrays, but with sightly different syntax on occasions.

In [ ]:
shape = (2,3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")


PyTorch has many pre-loaded datasets and those can be found using "torch.utils.data.DataLoader" and "torch.utils.data.Dataset". MNIST is a foundational dataset of hand-drawn didgits. It consists of 70,000 greyscale 28x28 images with 10 classes, 0 through 9. It is split into train and test sets, the 'train=True' line controls which part of the dataset is loaded.

## Task 1: Pre-process the data by:
* transforming the data into an torchvision.tv_tensors.Image with datatype float32 and scaled pixel values between [0,1]
* set a batch size
* choose how large the training set should be (hint: between 1 and 5999)

In [ ]:
from torch.utils.data import Dataset, random_split
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.transforms import v2
import torch.nn.functional as F

trainval_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(FILL_HERE, scale=FILL_HERE)])
)

train_size = FILL_HERE
val_size   = len(trainval_data) - train_size
train_data, val_data = random_split(trainval_data, [train_size, val_size], generator=torch.Generator().manual_seed(42))

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(FILL_HERE, scale=FILL_HERE)])
)

batch_size = FILL_HERE
train_dataloader = DataLoader(train_data, batch_size=batch_size)
val_dataloader = DataLoader(val_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

Visualize the data.

In [ ]:
labels_map = {
    0: "zero",
    1: "one",
    2: "two",
    3: "three",
    4: "four",
    5: "five",
    6: "six",
    7: "seven",
    8: "eight",
    9: "nine",
}

figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(train_data), size=(1,)).item()
    img, label = train_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

# Building the Neural Network

Start by importing "nn" from torch, this is where all building blocks for the neural networks are located.

Create a class 'NeuralNetwork', then define the function "\__init
__" to preprocess the data and set the layers in the network.

* "nn.Sequential" will create a module list that is treated as a single module and runs the entire list in descending order.
* "nn.Flatten" would reshape the input into a one dimensional tensor
* "nn.Linear" is the PyTorch implelemtation of a fully connected (dense) layer by applying a linear transformation (matrix multiplication and a bias term)
* "nn.Sogmoid" is an activation function that introduces nonlinearity
* "nn.ReLU" applies the rectified linear unit function (max{0,x}) in an element-wise manner

The function 'forward' will compute a forward pass through the network.


## Task 2: fill in the input size, hidden layer sizes, and num classes.

In [ ]:
from torch import nn

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(FILL_HERE, FILL_HERE),
            nn.ReLU(),
            nn.Linear(FILL_HERE), FILL_HERE),
            nn.ReLU(),
            nn.Linear(FILL_HERE, FILL_HERE)
        )

    def forward(self, x):
        logits = self.network(x)
        return logits

Create an instance of the model, print the layers, and calculate the number of trainable parameters.

In [ ]:
# Load model to device and print architecture
model = NeuralNetwork().to(device)
print(model)

# Count parameters in the entire model
total_params = sum(p.numel() for p in model.parameters())
print("\nTotal parameters:", total_params)

# Loss Functions and Optimizers

Loss functions compute the difference between the predictions and the ground truth values. Regression and classification require different types of loss functions, since the goal of the prediction is slightly different. How are they different?

Common regression loss functions:
* mean square error
* mean absolute error (L1)
* Huber

Common classification loss functions:
* binary cross entropy
* categorical cross entropy
* negative log-likelihood



Optimizers use the loss to update the model weights to find the lowest possible loss value for the model.

Common optimizers:
* gradient descent
* stochastic gradient descent
* Adam


## Task 3: choose reasonable loss functions and optimizers, given the target labels are integers.

In [ ]:
loss_fn = nn.FILL_HERE
optimizer = torch.optim.FILL_HERE

# Auto-grad

When training neural networks, the most frequently used algorithm is back propagation. In this algorithm, parameters are adjusted according to the gradient of the loss function with respect to the given parameter.

To compute those gradients, PyTorch has a built-in differentiation engine called "torch.autograd". You can set the value of "requires_grad" when creating a tensor, or later by using "x.requires_grad_(True)" method.

To optimize weights of parameters in the neural network, you need to compute the derivatives of the loss function with respect to parameters.
To compute those derivatives, call "loss.backward()".

Also, there are reasons to want to disable gradient tracking:
* to mark some parameters in your neural network as frozen parameters.
* to speed up computations when you are only doing forward pass, because computations on tensors that do not track gradients would be more efficient.



# Training/Testing Loop

Once the model is created and the loss/optimizer functions are chosen, the training can commence.
Train and eval loop functions are created so that model parameters are only updated when running with train data. Why is this important?




In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    correct = 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def eval_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    correct = 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    correct /= size
    print(f"Accuracy: {(correct):0.4f}\nLoss: {(loss):0.4f}\n")

## Task 4: choose the number of epochs and train the model

In [ ]:
epochs = FILL_HERE

for t in range(epochs):
    print(f"Epoch {t+1}\n")

    # Train model on train data (update params)
    train_loop(train_dataloader, model, loss_fn, optimizer)

    # Check train accuracy for this epoch
    print("\nTrain:")
    eval_loop(train_dataloader, model, loss_fn)

    # Check val accuracy for this epoch
    print("Val:")
    eval_loop(val_dataloader, model, loss_fn)

    print("-------------------------------\n")
print("Done!")

# Evaluating the Model

After training, the performance of the model can be further analyzed by breaking down the accuracy, precision, recall, and f1 score for each in the test set. Why might this be important?

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Put model in evaluation mode
model.eval()

all_y_true = []
all_y_pred = []

# Run test data through model again, freezing gradients since only doing a forward pass
with torch.no_grad():
  for X, y in test_dataloader:
    X, y = X.to(device), y.to(device)
    pred = model(X)
    # Get predicted class indices (argmax of logits)
    y_pred = pred.argmax(dim=1).cpu().numpy()
    y_true = y.cpu().numpy()

    all_y_pred.append(y_pred)
    all_y_true.append(y_true)


all_y_pred = np.concatenate(all_y_pred)
all_y_true = np.concatenate(all_y_true)

print(classification_report(all_y_true, all_y_pred))

In [ ]:
class_labels = np.unique(np.concatenate((y_true, y_pred)))
print(f"Detected {len(class_labels)} classes: {class_labels}")

# Compute confusion matrix
cm = confusion_matrix(all_y_true, all_y_pred, labels=class_labels)

# Plot annotated confusion matrix
plt.figure(figsize=(6,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cbar=True,
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.title("Confusion Matrix (Test)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()